In [ ]:
! unzip /content/Generated_Real_Anomalies_5.zip
! unzip /content/Healthy.zip

In [ ]:
! unzip /content/Final_Anomalous.zip
! unzip /content/Final_Normal.zip

unzip:  cannot find or open /content/Final_Anomalous.zip, /content/Final_Anomalous.zip.zip or /content/Final_Anomalous.zip.ZIP.
unzip:  cannot find or open /content/Final_Normal.zip, /content/Final_Normal.zip.zip or /content/Final_Normal.zip.ZIP.


# Creating dataloaders

In [ ]:
import numpy as np
import torch
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
import glob

def load_data(folders):
    data = []
    labels = []
    for label, folder in enumerate(folders):
        for csv_file in glob.glob(folder + '/*.csv'):
            df = pd.read_csv(csv_file)
            data.append(df.values)
            labels.append(label)
    data = np.array(data)
    return torch.tensor(data, dtype=torch.float32), torch.tensor(labels, dtype=torch.long)

def create_datasets_and_loaders(folders, batch_size, num_anomalous_train):
    data, labels = load_data(folders)

    # Split data by class
    healthy_data = data[labels == 0]
    anomalous_data = data[labels == 1]

    # Shuffle data to ensure randomness
    healthy_indices = torch.randperm(len(healthy_data))
    anomalous_indices = torch.randperm(len(anomalous_data))

    healthy_data = healthy_data[healthy_indices]
    anomalous_data = anomalous_data[anomalous_indices]

    # Reshape data to [N, 1, 500]
    healthy_data = healthy_data.reshape(len(healthy_data), 1, -1)
    anomalous_data = anomalous_data.reshape(len(anomalous_data), 1, -1)

    # Define test set sizes
    test_size = 600

    # Split data into training and test sets
    train_healthy = healthy_data[test_size:]  # All remaining healthy data goes to training
    test_healthy = healthy_data[:test_size]  # Fixed 600 healthy observations for test

    train_anomalous = anomalous_data[test_size:test_size + num_anomalous_train]  # Select the number of anomalous observations for training
    test_anomalous = anomalous_data[:test_size]  # Fixed 600 anomalous observations for test

    # Combine training data
    train_data = torch.cat([train_healthy, train_anomalous], dim=0)
    train_labels = torch.cat([torch.zeros(len(train_healthy)), torch.ones(len(train_anomalous))], dim=0).long()  # Ensure labels are of type torch.long

    # Combine test data
    test_data = torch.cat([test_healthy, test_anomalous], dim=0)
    test_labels = torch.cat([torch.zeros(len(test_healthy)), torch.ones(len(test_anomalous))], dim=0).long()  # Ensure labels are of type torch.long

    # Create datasets and loaders
    train_dataset = TensorDataset(train_data, train_labels)
    test_dataset = TensorDataset(test_data, test_labels)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, test_loader

# Define the folders and the number of anomalous observations for the training set
class_folders = [
    '/content/Healthy',
    '/content/Generated_Real_Anomalies_5'
]
num_anomalous_train = 5  # Specify the number of anomalous observations in the training set

train_loader, test_loader = create_datasets_and_loaders(class_folders, batch_size=16, num_anomalous_train=num_anomalous_train)

In [ ]:
def count_labels(data_loader):
    label_counts = {}
    for _, labels in data_loader:
        for label in labels:
            label = int(label.item())  # Convert tensor to integer
            if label in label_counts:
                label_counts[label] += 1
            else:
                label_counts[label] = 1
    return label_counts

# Example usage
train_label_counts = count_labels(train_loader)
test_label_counts = count_labels(test_loader)

print("Train Label Counts:", train_label_counts)
print("Test Label Counts:", test_label_counts)

Train Label Counts: {0: 2400, 1: 1}
Test Label Counts: {0: 600, 1: 600}


In [ ]:
data, labels = next(iter(train_loader))


print(f'Data dimensions: {data.shape}')
print(f'Labels dimensions: {labels.shape}')

Data dimensions: torch.Size([16, 1, 500])
Labels dimensions: torch.Size([16])


# CNN Model creation

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.downsample = downsample

    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = F.relu(self.bn1(out))
        out = self.conv2(out)
        out = self.bn2(out)
        if self.downsample:
            residual = self.downsample(x)
        out += residual
        out = F.relu(out)
        return out


class ResNet(nn.Module):
    def __init__(self, block, layers, num_classes=2):
        super(ResNet, self).__init__()
        self.in_channels = 64
        self.conv = nn.Conv1d(1, 64, kernel_size=7, stride=2, padding=3)
        self.bn = nn.BatchNorm1d(64)
        self.layer1 = self.make_layer(block, 64, layers[0])
        self.layer2 = self.make_layer(block, 128, layers[1], 2)
        self.layer3 = self.make_layer(block, 256, layers[2], 2)
        self.layer4 = self.make_layer(block, 512, layers[3], 2)
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(512, num_classes)

    def make_layer(self, block, out_channels, blocks, stride=1):
        downsample = None
        if (stride != 1) or (self.in_channels != out_channels):
            downsample = nn.Sequential(
                nn.Conv1d(self.in_channels, out_channels, kernel_size=1, stride=stride),
                nn.BatchNorm1d(out_channels))
        layers = []
        layers.append(block(self.in_channels, out_channels, stride, downsample))
        self.in_channels = out_channels
        for i in range(1, blocks):
            layers.append(block(out_channels, out_channels))
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn(self.conv(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.avg_pool(out)
        out = out.view(out.size(0), -1)
        out = self.fc(out)
        return out


In [ ]:
from sklearn.metrics import precision_recall_fscore_support
def train_model(model, train_loader, test_loader, device, num_epochs=10):
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)
        #print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}')

    model.eval()
    correct = 0
    total = 0
    all_labels = []
    all_predictions = []

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predicted.cpu().numpy())

    accuracy = 100 * correct / total
    #print(f'Accuracy of the model on the test set: {accuracy}%')

    precision, recall, f1_score, _ = precision_recall_fscore_support(all_labels, all_predictions, average='weighted')
    #print(f'F1 Score: {f1_score:.4f}')
    return accuracy, f1_score

model = ResNet(ResidualBlock, [2, 2, 2, 2], num_classes=2)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.to(device)

ResNet(
  (conv): Conv1d(1, 64, kernel_size=(7,), stride=(2,), padding=(3,))
  (bn): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (layer1): Sequential(
    (0): ResidualBlock(
      (conv1): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(1,))
      (bn1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(1,))
      (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): ResidualBlock(
      (conv1): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(1,))
      (bn1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(1,))
      (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
  )
  (layer2): Sequential(
    (0): ResidualBlock(
      (conv1): Conv1d(64, 1

In [ ]:
train_model(model, train_loader, test_loader, device, num_epochs=1)

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


50.0

# Multistart CNN

In [ ]:
accur = []
score = []
for i in range(30):
    num_anomalous_train = 2000
    train_loader, test_loader = create_datasets_and_loaders(class_folders, batch_size=16, num_anomalous_train=num_anomalous_train)
    model = ResNet(ResidualBlock, [2, 2, 2, 2], num_classes=2)
    model.to(device)
    accuracy, f_score = train_model(model, train_loader, test_loader, device, num_epochs=2)
    accur.append(accuracy)
    score.append(f_score)
    print(i)



0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29


In [ ]:
results

[100.0, 99.75, 100.0, 99.75, 100.0]

In [ ]:
# Calculate the mean
mean_ac = np.mean(accur)
mean_score = np.mean(score)

# Calculate the standard deviation
std_dev_acc = np.std(accur)
std_dev_score = np.std(score)


# Display the results
print(f"Mean: {mean}")
print(f"Standard Deviation: {std_dev}")
print(f"Mean: {mean_score}")
print(f"Standard Deviation: {std_dev_score}")

Mean: 99.98611111111111
Standard Deviation: 0.043479099562496765
Mean: 0.9912939832228514
Standard Deviation: 0.04352492631828981


# MLP

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Define the MLP model for binary classification
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.hidden = nn.Linear(500, 46)  # Assuming the input size is 500 as per your data reshaping
        self.activation = nn.Tanh()  # tan-sigmoid activation function
        self.output = nn.Linear(46, 1)  # Output layer for binary classification

    def forward(self, x):
        x = self.hidden(x)
        x = self.activation(x)
        x = self.output(x)
        return torch.sigmoid(x)  # Sigmoid activation for binary classification

# Instantiate the model
model = MLP()

# Define the loss function and optimizer
criterion = nn.BCELoss()  # Binary Cross Entropy Loss for binary classification
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Adam optimizer

# Training loop with test accuracy evaluation
def train_model(model, train_loader, test_loader, criterion, optimizer, num_epochs=20):
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            # Flatten the input tensor
            inputs = inputs.view(inputs.size(0), -1)

            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, labels.float().unsqueeze(1))

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        # Calculate and print the average loss for this epoch
        avg_train_loss = running_loss / len(train_loader)
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_train_loss:.4f}')

        # Evaluate on test data
        test_accuracy = evaluate_model(model, test_loader)
        print(f'Epoch [{epoch+1}/{num_epochs}], Test Accuracy: {test_accuracy:.2f}%')

# Function to evaluate the model on the test set
def evaluate_model(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            # Flatten the input tensor
            inputs = inputs.view(inputs.size(0), -1)

            # Forward pass
            outputs = model(inputs)
            predicted = (outputs > 0.5).float()  # Apply threshold of 0.5
            total += labels.size(0)
            correct += (predicted.squeeze() == labels).sum().item()

    accuracy = 100 * correct / total
    return accuracy

# Train the model
train_model(model, train_loader, test_loader, criterion, optimizer, num_epochs=20)

Epoch [1/20], Loss: 0.0698
Epoch [1/20], Test Accuracy: 50.00%
Epoch [2/20], Loss: 0.0288
Epoch [2/20], Test Accuracy: 50.00%
Epoch [3/20], Loss: 0.0272
Epoch [3/20], Test Accuracy: 50.00%
Epoch [4/20], Loss: 0.0270
Epoch [4/20], Test Accuracy: 50.00%
Epoch [5/20], Loss: 0.0269
Epoch [5/20], Test Accuracy: 50.00%
Epoch [6/20], Loss: 0.0270
Epoch [6/20], Test Accuracy: 50.00%
Epoch [7/20], Loss: 0.0270
Epoch [7/20], Test Accuracy: 50.00%
Epoch [8/20], Loss: 0.0270
Epoch [8/20], Test Accuracy: 50.00%
Epoch [9/20], Loss: 0.0271
Epoch [9/20], Test Accuracy: 50.00%
Epoch [10/20], Loss: 0.0271
Epoch [10/20], Test Accuracy: 50.00%
Epoch [11/20], Loss: 0.0272
Epoch [11/20], Test Accuracy: 50.00%
Epoch [12/20], Loss: 0.0272
Epoch [12/20], Test Accuracy: 50.00%
Epoch [13/20], Loss: 0.0272
Epoch [13/20], Test Accuracy: 50.00%


KeyboardInterrupt: 

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Define the MLP model
class MLP(nn.Module):
    def __init__(self, input_size):
        super(MLP, self).__init__()
        self.hidden = nn.Linear(input_size, 46)  # Hidden layer with 46 neurons
        self.output = nn.Linear(46, 2)  # Output layer with 2 neurons for binary classification
        self.activation = nn.Tanh()  # Using tan-sigmoid (Tanh) activation function

    def forward(self, x):
        x = x.view(x.size(0), -1)  # Flatten the input to (batch_size, input_size)
        x = self.activation(self.hidden(x))
        x = self.output(x)  # Raw logits for CrossEntropyLoss
        return x

# Initialize the model, loss function, and optimizer
input_size = 500  # Assuming input data is reshaped to [N, 1, 500]
model = MLP(input_size)
criterion = nn.CrossEntropyLoss()  # Cross-Entropy Loss for 2 output neurons
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
def train_model(model, train_loader, criterion, optimizer, num_epochs=10):
    model.train()
    for epoch in range(num_epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            # Move data to the appropriate device
            inputs, labels = inputs.cuda(), labels.cuda()

            # Zero the parameter gradients
            optimizer.zero_grad()

            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            # Backward pass and optimization
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {running_loss / len(train_loader)}")

# Testing loop
def test_model(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            # Move data to the appropriate device
            inputs, labels = inputs.cuda(), labels.cuda()

            # Forward pass
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)  # Take the index of the max logit as the prediction
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print(f'Accuracy: {100 * correct / total:.2f}%')

# Train and test the model
model.cuda()  # Move the model to GPU if available
train_model(model, train_loader, criterion, optimizer, num_epochs=10)
test_model(model, test_loader)

Epoch 1/10, Loss: 0.041341994577611715
Epoch 2/10, Loss: 0.02711896523093151
Epoch 3/10, Loss: 0.02725456500524618
Epoch 4/10, Loss: 0.027310369713601984
Epoch 5/10, Loss: 0.027135845179675728
Epoch 6/10, Loss: 0.027125511765603395
Epoch 7/10, Loss: 0.027286244002756792
Epoch 8/10, Loss: 0.02716207784286003
Epoch 9/10, Loss: 0.027534790269241032
Epoch 10/10, Loss: 0.02758835342601682
Accuracy: 50.00%


# SVM

In [ ]:
import torch
from sklearn import svm
from sklearn.metrics import accuracy_score

# Function to convert a DataLoader to numpy arrays
def dataloader_to_numpy(dataloader):
    all_data = []
    all_labels = []
    for inputs, labels in dataloader:
        all_data.append(inputs.numpy().reshape(inputs.size(0), -1))  # Flatten inputs
        all_labels.append(labels.numpy())
    return np.concatenate(all_data), np.concatenate(all_labels)

# Convert train_loader to numpy arrays

train_data, train_labels = dataloader_to_numpy(train_loader)
print(train_data.shape)
print(train_labels.shape)
# Convert test_loader to numpy arrays
test_data, test_labels = dataloader_to_numpy(test_loader)

# Initialize and train the SVM model
svm_model = svm.SVC(kernel='poly')  # You can choose different kernels, e.g., 'rbf', 'poly', etc.
svm_model.fit(train_data, train_labels)

# Predict on the test data
test_predictions = svm_model.predict(test_data)

# Calculate and print the accuracy
accuracy = accuracy_score(test_labels, test_predictions)
print(f'Accuracy: {accuracy * 100:.2f}%')

(2401, 500)
(2401,)
Accuracy: 57.17%


In [ ]:
result=[]
for i in range(30):
  num_anomalous = 2400
  train_loader, test_loader = create_datasets_and_loaders(class_folders, batch_size=16, num_anomalous_train=num_anomalous)
  train_data, train_labels = dataloader_to_numpy(train_loader)
  test_data, test_labels = dataloader_to_numpy(test_loader)
  # Initialize and train the SVM model
  svm_model = svm.SVC(kernel='poly')  # You can choose different kernels, e.g., 'rbf', 'poly', etc.
  svm_model.fit(train_data, train_labels)
  # Predict on the test data
  test_predictions = svm_model.predict(test_data)
  # Calculate and print the accuracy
  accuracy = accuracy_score(test_labels, test_predictions)
  result.append(accuracy)
  print(i)


0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29


In [ ]:
# Calculate the mean
mean = np.mean(result)

# Calculate the standard deviation
std_dev = np.std(result)

# Display the results
print(f"Mean: {mean}")
print(f"Standard Deviation: {std_dev}")

Mean: 0.9993888888888889
Standard Deviation: 0.0008588680463188979
